# Workshop: Aprendizado de máquina aplicado à Biologia

## **Considerações iniciais**

Este notebook acompanha o workshop: cada seção traz uma explicação breve, seguida do código. Execute as células de cima para baixo, porque cada parte usa variáveis criadas nas células anteriores.

O workshop tem três partes, cada uma com um *dataset*:

- **Parte 1 - Regressão linear:** *Diabetes*, com 442 pacientes.
- **Parte 2 - Classificação:** *Breast Cancer Wisconsin*, com 569 nódulos de mama.
- **Parte 3 - Métodos não supervisionados:** o mesmo *Breast Cancer Wisconsin*, agora sem o diagnóstico.

## **Preparação do ambiente**

Python tem um grande ecossistema de **bibliotecas**: pacotes de código prontos e testados, que importamos em vez de programar tudo do zero. Neste workshop, usamos:

- **NumPy** (`np`): operações numéricas com vetores e matrizes;
- **pandas** (`pd`): tabelas de dados (*DataFrames*), parecidas com planilhas;
- **Matplotlib** (`plt`): gráficos, com controle de cada elemento;
- **seaborn** (`sns`): gráficos estatísticos prontos, construídos sobre o Matplotlib;
- **scikit-learn** (`sklearn`): modelos de aprendizado de máquina e ferramentas de avaliação, importados ao longo do notebook, perto de onde são usados.

A variável `SEED` fixa a semente dos sorteios aleatórios, como a escolha de quais pacientes vão para o treino. Assim, todos que executarem o notebook obtêm os mesmos resultados.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42  # semente usada em todas as divisões e modelos, para resultados reprodutíveis

A célula abaixo define a paleta de cores e a aparência dos gráficos. Não é preciso alterá-la.

In [ ]:
# Configuração visual dos gráficos (paleta do material)
from cycler import cycler
from matplotlib.colors import LinearSegmentedColormap

CORES = {
    "papel": "#f8f4ea",
    "texto": "#332d28",
    "texto_secundario": "#62584f",
    "linha": "#d9cfbe",
    "azul": "#435d73",
    "verde": "#2d4038",
    "ocre": "#90682e",
    "azul_acinzentado": "#6b7c90",
    "preenchimento_azul": "#e9eef2",
    "preenchimento_creme": "#f4e6cd",
}

CMAP_CORRELACAO = LinearSegmentedColormap.from_list(
    "azul_papel_ocre", [CORES["azul"], CORES["papel"], CORES["ocre"]]
)
CMAP_CONTAGEM = LinearSegmentedColormap.from_list("papel_azul", [CORES["papel"], CORES["azul"]])

ESTILO_PONTOS = {"alpha": 0.6, "edgecolor": CORES["papel"], "linewidth": 0.5}

ESTILO_BOXPLOT = {
    "color": CORES["preenchimento_azul"],
    "linecolor": CORES["azul"],
    "medianprops": {"color": CORES["azul"], "linewidth": 2},
    "showfliers": False,
}

plt.rcParams.update({
    "figure.figsize": (6, 4),
    "figure.dpi": 110,
    "figure.facecolor": CORES["papel"],
    "savefig.facecolor": CORES["papel"],
    "axes.facecolor": CORES["papel"],
    "axes.edgecolor": CORES["linha"],
    "axes.labelcolor": CORES["texto"],
    "axes.titlecolor": CORES["texto"],
    "axes.prop_cycle": cycler(color=[
        CORES["azul"], CORES["verde"], CORES["ocre"], CORES["azul_acinzentado"]
    ]),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "axes.axisbelow": True,
    "grid.color": CORES["linha"],
    "grid.linewidth": 0.8,
    "patch.edgecolor": CORES["papel"],
    "lines.linewidth": 2,
    "text.color": CORES["texto"],
    "xtick.color": CORES["texto_secundario"],
    "ytick.color": CORES["texto_secundario"],
    "legend.facecolor": CORES["papel"],
    "legend.edgecolor": CORES["linha"],
    "legend.framealpha": 1,
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans"],
    "font.size": 11,
    "axes.titlesize": 13,
})

# **Parte 1 - Conceitos gerais e regressão linear**

## **1.1 Problema biológico**

> **Pergunta central:** o modelo consegue prever uma medida da progressão do diabetes, um ano após a avaliação inicial, a partir das características dos pacientes?

<p align="center">
  <img src="https://raw.githubusercontent.com/lucasofalonso/workshop-ml-biologia/main/images/dataset_diabetes.png" alt="Dos pacientes aos dados" width="75%"><br>
  <em><b>Figura 1.</b> Ilustração do <i>dataset</i> Diabetes.</em>
</p>

Para responder a essa pergunta, usaremos o *dataset* Diabetes, ilustrado na Figura 1. Ele reúne a avaliação inicial de **442 pacientes**, com **10 características** registradas para cada um: idade, sexo, índice de massa corporal (IMC), pressão arterial média e seis medidas do sangue (`s1` a `s6`), como colesterol, triglicerídeos e glicose. Um ano depois, a progressão da doença foi medida e registrada como um número: quanto maior, mais a doença avançou. Cada paciente ocupa uma linha da tabela.

**Regressão linear: uma bancada de especialistas.** Digamos que você queira avaliar o preço do seu carro. É natural procurar alguém que entenda de carros. Esse especialista considera as **características** do carro e atribui a cada uma um **peso**: a quilometragem tende a baixar o preço; a economia de combustível, a valorizá-lo. A regressão linear funciona assim: a previsão é uma **soma ponderada**, um valor de partida somado às contribuições de cada característica, em que cada contribuição é o valor da característica multiplicado pelo seu peso:

$$\text{preço} = w_0 + w_1 \cdot \text{quilometragem} + w_2 \cdot \text{economia} + \dots$$

O valor de partida $w_0$ é o **intercepto**. Cada peso diz o quanto a característica influencia a previsão e em que direção: aqui, $w_1$ seria negativo e $w_2$, positivo. De forma geral, com $p$ características $x_1, \dots, x_p$, a previsão $\hat{y}$ (lê-se "y chapéu") é:

$$\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + \dots + w_p x_p$$

**Aprendizado supervisionado.** O especialista aprende com a experiência, e o modelo também. Apresentamos exemplos de carros com preços conhecidos, e ele ajusta os pesos para que suas estimativas se aproximem desses preços. É como uma criança aprendendo a andar de bicicleta com rodinhas: as respostas conhecidas apoiam o aprendizado. Ter exemplos acompanhados das respostas é o que caracteriza o aprendizado **supervisionado**.

Depois do treinamento, tiramos as rodinhas: apresentamos um carro novo e perguntamos o preço estimado. A criança caiu ou aprendeu a andar? Por isso, avaliamos o modelo em exemplos que ele não viu e, mais adiante, dividiremos os dados em treino, validação e teste.

A regressão linear sempre prevê um número, como um preço. Estimar uma quantidade é o que chamamos de **regressão**. Quando a resposta é uma categoria, como benigno ou maligno na Parte 2, falamos em **classificação**. Na Parte 3, veremos o aprendizado **não supervisionado**, em que não há respostas conhecidas.

No nosso problema, os pacientes fazem o papel dos carros: o modelo atribui pesos às características de cada paciente para prever a progressão da doença um ano depois.

## **1.2 Carregando os dados**

Agora vamos tratar de um objeto central em aprendizado de máquina: o *dataset*, ou simplesmente **conjunto de dados**. Ele reúne os exemplos que utilizaremos para treinar e avaliar o modelo. Na Figura 2, podemos ver os elementos que o compõem.

<p align="center">
  <img src="https://raw.githubusercontent.com/lucasofalonso/workshop-ml-biologia/main/images/composição_dataset.png" alt="Componentes de um dataset" width="75%"><br>
  <em><b>Figura 2.</b> Esquema ilustrativo dos componentes de um <i>dataset</i>.</em>
</p>

Aqui, trabalharemos com **dados tabulares**, organizados em linhas e colunas. Cada linha corresponde a um exemplo, neste caso, um paciente. Cada coluna de entrada corresponde a uma **característica** desse paciente, como sua idade ou seu índice de massa corporal. Em aprendizado de máquina, essas características de entrada são chamadas de *features*. Reunidas em uma tabela numérica, elas formam a **matriz de *features***, que chamaremos de `X`.

Além das *features*, nosso *dataset* contém uma **resposta conhecida para cada paciente**: a medida da progressão do diabetes um ano após a avaliação inicial. Essa variável é o *target*, o alvo que queremos aprender a prever. Os valores observados do *target* formam o *target vector*, representado por `y`. Por ora, podemos entendê-lo como uma lista: o primeiro valor é a resposta observada para o primeiro paciente, o segundo, para o segundo paciente, e assim por diante. Essa ordem acompanha as linhas de `X`.

**Cada paciente tem, portanto, suas características de entrada e uma resposta conhecida associada a elas.** É essa disponibilidade de exemplos com respostas conhecidas que caracteriza o aprendizado supervisionado.

No código, armazenaremos `X` em um `DataFrame` do pandas, uma estrutura que permite trabalhar com tabelas preservando os nomes das colunas e a identificação das linhas. Já `y` ficará em uma `Series`, uma estrutura para trabalhar com uma sequência de valores, também identificados por rótulos.

Na literatura, um mesmo conceito pode receber nomes diferentes:

- **Exemplo:** uma unidade do conjunto de dados, aqui um paciente. Também chamado de observação ou instância.
- ***Feature*:** uma característica de entrada, como idade ou IMC. Também chamada de atributo.
- **Matriz de *features* (`X`):** um paciente por linha e uma *feature* por coluna.
- ***Target*:** a variável que queremos prever.
- ***Target vector* (`y`):** os valores observados do *target*, na mesma ordem dos pacientes em `X`.

In [ ]:
from sklearn.datasets import load_diabetes

dataset = load_diabetes(as_frame=True, scaled=False)

X = dataset.data                                     # features: uma linha por paciente
y = dataset.target.rename("disease_progression")    # target: progressão um ano depois

## **1.3 Exploração inicial**

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
X.info()

In [ ]:
X["bmi"].value_counts()

In [ ]:
# sex está armazenado como número, mas representa duas categorias
X["sex"].value_counts(normalize=True) * 100

In [ ]:
X_cat_cols = ["sex"]
X_num_cols = [col for col in X.columns if col not in X_cat_cols]

print("Numéricas:", X_num_cols)
print("Categóricas:", X_cat_cols)

## **1.4 Ferramentas estatísticas**

**Uma estatística resume, em um número, algum aspecto do conjunto de dados.**

<p align="center">
  <img src="https://raw.githubusercontent.com/lucasofalonso/workshop-ml-biologia/main/images/ferramentas_estatisticas.png" alt="Perguntas e ferramentas estatísticas" width="75%"><br>
  <em><b>Figura 3.</b> Perguntas orientam a escolha das ferramentas estatísticas.</em>
</p>

- O *dataset* como uma caixa preta com compartimentos; cada pergunta abre um, e as estatísticas são as chaves.
- Qual é o valor típico? Quanto os valores variam? Onde se concentram? Como as variáveis se relacionam?

### **Resumo estatístico**

In [ ]:
X[X_num_cols].describe()

In [ ]:
y.describe()

- `count`: valores disponíveis; `mean`: média; `std`: desvio padrão.
- `min` e `max`: menor e maior valor.
- `25%`, `50%` e `75%`: quartis; o `50%` é a mediana.

### **Valor típico: média, mediana e moda**

In [ ]:
pd.DataFrame({
    "Média": [y.mean()],
    "Mediana": [y.median()],
    "Modas": [y.mode().to_list()],
}).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(y.index, y, s=18, color=CORES["azul"], **ESTILO_PONTOS)
ax.axhline(y.median(), color=CORES["verde"], label=f"Mediana = {y.median():.1f}")
for i, moda in enumerate(y.mode()):
    ax.axhline(moda, color=CORES["azul_acinzentado"], linestyle=":", label="Moda" if i == 0 else None)
ax.scatter(y.index.values.mean(), y.mean(), marker="x", s=160, linewidths=3,
           color=CORES["ocre"], zorder=3, label=f"Média = {y.mean():.1f}")
ax.set(xlabel="Paciente", ylabel="Progressão da doença")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1), ncols=3, frameon=False)
fig.tight_layout()
plt.show()

- **Média** (x): centro de massa dos pontos; valores extremos a puxam em sua direção.
- **Mediana**: centro dos dados ordenados; pouco sensível aos extremos.
- **Moda**: valor mais frequente; aqui, duas modas (72 e 200).
- Média (152,1) acima da mediana (140,5): sinal de valores altos puxando a média.

### **Dispersão: desvio padrão e *boxplot***

In [ ]:
media, desvio = y.mean(), y.std()
q1, q2, q3 = y.quantile([0.25, 0.5, 0.75])

fig, (ax_pontos, ax_box) = plt.subplots(1, 2, sharey=True, figsize=(8, 4.5), width_ratios=[3, 1])
ax_pontos.axhspan(media - desvio, media + desvio, color=CORES["preenchimento_creme"], zorder=0,
                  label=f"Média ± desvio padrão ({desvio:.1f})")
ax_pontos.scatter(y.index, y, s=18, color=CORES["azul"], **ESTILO_PONTOS)
ax_pontos.axhline(media, color=CORES["ocre"], linestyle="--")
ax_pontos.set(xlabel="Paciente", ylabel="Progressão da doença")
ax_pontos.legend(loc="lower center", bbox_to_anchor=(0.5, 1), frameon=False)

sns.boxplot(y=y, width=0.4, ax=ax_box, **ESTILO_BOXPLOT)
for nome, valor in [("Q1", q1), ("Q2", q2), ("Q3", q3)]:
    ax_box.annotate(f"{nome} = {valor:.1f}", (0.2, valor), xytext=(6, 0), textcoords="offset points", va="center")
ax_box.set(title=f"IQR = {q3 - q1:.1f}", ylabel="", xticks=[])
ax_box.grid(False)
fig.tight_layout()
plt.show()

- **IQR** = Q3 − Q1 ≈ 124,5: largura da metade central dos pacientes, a altura da caixa.
- Hastes até 1,5 IQR além da caixa; não há pontos isolados neste *dataset*.
- **Desvio padrão** ≈ 77,1: distância típica dos valores até a média; sensível a extremos, como a própria média.

### **Forma da distribuição: histograma, probabilidade e KDE**

In [ ]:
limites = [0, 50, 100, 150, 200, 250, 300, 350]
caixas = pd.cut(y, bins=limites, right=False)

pd.DataFrame({
    "Pacientes": caixas.value_counts(sort=False),
    "Frequência relativa": caixas.value_counts(normalize=True, sort=False).round(4),
})

- Intervalo `[100, 150)`: inclui 100, exclui 150.
- Frequência relativa = probabilidade de sortear, entre os 442, um paciente naquela faixa.
- A amostra estima a população que queremos estudar: inferência estatística.

In [ ]:
# Altura de cada ponto dentro da sua caixa: 0.5, 1.5, 2.5, ... (um paciente por "andar")
caixa_de_cada = np.digitize(y, limites[:-1])
altura = y.groupby(caixa_de_cada).cumcount() + 0.5

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.histplot(y, bins=limites, color=CORES["preenchimento_azul"], edgecolor=CORES["azul"], alpha=1, ax=axes[0])
axes[0].scatter(y, altura, s=8, color=CORES["azul"], linewidths=0)
axes[0].set(title="Frequência absoluta", xlabel="Progressão da doença", ylabel="Pacientes")

sns.histplot(y, bins=limites, stat="probability", color=CORES["azul"], alpha=0.75, ax=axes[1])
for barra in axes[1].patches:
    axes[1].annotate(f"{barra.get_height():.0%}", (barra.get_x() + barra.get_width() / 2, barra.get_height()),
                     xytext=(0, 3), textcoords="offset points", ha="center", fontsize=9)
axes[1].set(title="Frequência relativa", xlabel="Progressão da doença", ylabel="Fração dos pacientes")

fig.tight_layout()
plt.show()

- Esquerda: cada ponto é um paciente, empilhado na sua caixa; a altura da barra é a contagem.
- Direita: mesma forma, agora em fração dos 442 pacientes; as barras somam 100%.

In [ ]:
inicio, fim = 100, 150
selecao = (y >= inicio) & (y < fim)
quantidade = selecao.sum()
probabilidade = selecao.mean()

fig, ax = plt.subplots(figsize=(8.5, 4.8), dpi=120)
sns.histplot(
    x=y, bins=limites, stat="density",
    color=CORES["preenchimento_azul"], edgecolor=CORES["azul"], alpha=1, ax=ax,
)

for barra in ax.patches:
    if np.isclose(barra.get_x(), inicio):
        barra.set_facecolor(CORES["verde"])
        barra.set_edgecolor(CORES["verde"])
        destaque = barra

texto_probabilidade = f"{100 * probabilidade:.1f}".replace(".", ",")
ax.annotate(
    f"{quantidade} de {len(y)} pacientes\nÁrea destacada: {texto_probabilidade}%",
    xy=(destaque.get_x() + destaque.get_width() / 2, destaque.get_height()),
    xytext=(175, 0.9 * ax.get_ylim()[1]),
    va="center",
    arrowprops={"arrowstyle": "-|>", "color": CORES["texto"], "linewidth": 1.2, "connectionstyle": "arc3,rad=0.25"},
    bbox={"boxstyle": "round", "facecolor": CORES["papel"], "edgecolor": CORES["linha"]},
)

ax.set(
    title="Ao sortear um paciente do conjunto, qual é a chance de \nobservar uma progressão da doença de pelo menos 100 e menor que 150?",
    xlabel="Medida de progressão da doença",
    ylabel="Densidade",
)
fig.tight_layout()
plt.show()

- A frequência relativa depende da largura da caixa; a **densidade** = frequência relativa ÷ largura.
- Com densidade, a **área** da barra é a probabilidade: `[100, 150)` reúne 91 de 442 pacientes, ≈ 20,6%.
- Com mais observações e caixas mais estreitas, o histograma se aproxima de uma curva suave (dados simulados).

In [ ]:
from scipy import stats

# Dados simulados de uma distribuição normal conhecida (curva verde)
valores_simulados = np.random.default_rng(SEED).normal(loc=0, scale=1, size=4000)
dominio = (-4, 4)
x_curva = np.linspace(*dominio, 400)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharex=True, sharey=True)
for ax, (n_obs, n_caixas) in zip(axes, [(40, 6), (400, 18), (4000, 54)]):
    sns.histplot(valores_simulados[:n_obs], bins=np.linspace(*dominio, n_caixas + 1), stat="density",
                 color=CORES["preenchimento_azul"], edgecolor=CORES["azul"], alpha=1, ax=ax)
    ax.plot(x_curva, stats.norm.pdf(x_curva), color=CORES["verde"])
    ax.set(title=f"{n_obs:,} observações | {n_caixas} caixas".replace(",", "."),
           xlabel="Valor simulado", ylabel="Densidade", xlim=dominio)
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(y, bins=limites, stat="density", color=CORES["azul"], alpha=0.75, label="Histograma", ax=ax)
sns.kdeplot(y, cut=0, color=CORES["verde"], label="KDE", ax=ax)
ax.set(xlabel="Progressão da doença", ylabel="Densidade")
ax.legend()
fig.tight_layout()
plt.show()

- KDE: estimativa suave da densidade, direto dos dados; a área sob a curva também estima probabilidades.
- Cauda à direita: valores altos de progressão, os mesmos que puxam a média acima da mediana.

### **União das ferramentas estatísticas**

- Valor típico (média, mediana), variação (IQR, ± desvio padrão) e concentração (histograma, KDE) no mesmo eixo.

In [ ]:
media, mediana, desvio = y.mean(), y.median(), y.std()
q1, q3 = y.quantile([0.25, 0.75])

fig, (ax_box, ax_hist) = plt.subplots(2, 1, sharex=True, figsize=(8, 5.5), height_ratios=[1.4, 4])

sns.boxplot(x=y, width=0.4, ax=ax_box, **ESTILO_BOXPLOT)
ax_box.annotate("", xy=(q3, -0.34), xytext=(q1, -0.34),
                arrowprops={"arrowstyle": "<->", "color": CORES["verde"], "linewidth": 2, "shrinkA": 0, "shrinkB": 0})
ax_box.annotate(f"IQR = {q3 - q1:.1f}", ((q1 + q3) / 2, -0.34), xytext=(0, 3), textcoords="offset points",
                ha="center", va="bottom")
ax_box.grid(False)
ax_box.set(xlabel="", title="Distribuição da progressão da doença")

ax_hist.axvspan(media - desvio, media + desvio, color=CORES["preenchimento_creme"], zorder=0,
                label="Média ± desvio padrão")
sns.histplot(y, bins=limites, stat="density", color=CORES["azul"], alpha=0.75, label="Histograma", ax=ax_hist)
sns.kdeplot(y, cut=0, color=CORES["verde"], label="KDE", ax=ax_hist)
ax_hist.axvline(media, color=CORES["ocre"], label=f"Média = {media:.1f}")
ax_hist.axvline(mediana, color=CORES["verde"], linestyle="-.", label=f"Mediana = {mediana:.1f}")
for valor in (media - desvio, media + desvio):
    ax_hist.axvline(valor, color=CORES["ocre"], linestyle="--", linewidth=1.5)
ax_hist.set(xlabel="Progressão da doença", ylabel="Densidade")
ax_hist.legend(fontsize=9)

fig.tight_layout()
plt.show()

- Valores concentrados nas faixas baixas, com cauda à direita até 346.
- A cauda puxa a média (152,1) acima da mediana (140,5).
- Na caixa, a mediana fica mais perto de Q1 que de Q3: a mesma assimetria.

### **Relações entre variáveis: correlação de Pearson**

> **Pergunta:** como as características dos pacientes se relacionam com a progressão da doença?

<p align="center">
  <img src="https://raw.githubusercontent.com/lucasofalonso/workshop-ml-biologia/main/images/sombra_magica.png" alt="Analogia da sombra mágica" width="85%"><br>
  <em><b>Figura 4.</b> A sombra mágica: crescimento linear (A) e quadrático (B).</em>
</p>

- **(A)** A cada passo, a sombra cresce sempre o mesmo tanto: correlação linear positiva perfeita.
- **(B)** A altura depende do quadrado da distância: relação perfeita, mas o *r* de Pearson é zero.
- O *r* vai de −1 a 1: o sinal indica a direção da tendência; a proximidade de ±1, a intensidade.

In [ ]:
r_bmi = X["bmi"].corr(y)
inclinacao, intercepto = np.polyfit(X["bmi"], y, deg=1)
x_reta = np.array([X["bmi"].min(), X["bmi"].max()])

fig, ax = plt.subplots()
ax.scatter(X["bmi"], y, s=18, color=CORES["azul"], **ESTILO_PONTOS)
ax.plot(x_reta, intercepto + inclinacao * x_reta, color=CORES["ocre"], linestyle="--", label="Tendência linear")
ax.set(title=f"r = {r_bmi:.2f}", xlabel="Índice de massa corporal", ylabel="Progressão da doença")
ax.legend()
fig.tight_layout()
plt.show()

- IMC: *r* ≈ 0,59. Há uma tendência positiva, mas a nuvem é difusa: o IMC sozinho deixa muito sem explicar.
- *r*² ≈ 0,34: cerca de um terço da variação da progressão acompanha a tendência linear do IMC; o restante fica de fora. Guarde essa ideia: é a **variância explicada**.

In [ ]:
corr = X[X_num_cols].join(y).corr()

fig, axes = plt.subplots(1, 2, figsize=(13, 5), width_ratios=[1, 1.4])
corr_target = corr[y.name].drop(y.name).sort_values()
axes[0].barh(corr_target.index, corr_target.values, color=CORES["azul"])
axes[0].axvline(0, color=CORES["texto_secundario"], linewidth=0.8)
axes[0].grid(False)
axes[0].grid(axis="x")
axes[0].set(title="Correlação com a progressão", xlabel="r de Pearson")

sns.heatmap(corr, annot=True, fmt=".2f", annot_kws={"size": 8}, cmap=CMAP_CORRELACAO,
            vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title("Matriz de correlação")
fig.tight_layout()
plt.show()

- Mais associadas à progressão: `bmi`, `s5` (triglicerídeos), `bp` (pressão) e `s4`.
- `s3` (HDL) tem a única correlação negativa: quanto maior o HDL, menor tende a ser a progressão.
- Entre *features*: `s1` e `s2` (*r* = 0,90) e `s3` e `s4` (*r* = −0,74) contam parte da mesma história.
- **Correlação não é causalidade:** vendas de sorvete e afogamentos sobem juntos no verão. Veja também o [*Spurious Correlations*](https://tylervigen.com/spurious-correlations).

In [ ]:
# A única feature categórica: progressão por grupo de sexo
y.groupby(X["sex"]).agg(["mean", "median", "std", "count"]).round(2)

- Por sexo, médias de 149 e 156 e medianas de 140 e 141: diferença pequena quando olhamos `sex` isoladamente.

## **1.5 Divisão do *dataset***

<p align="center">
  <img src="https://raw.githubusercontent.com/lucasofalonso/workshop-ml-biologia/main/images/divisao_dataset.png" alt="Divisão do dataset em treino, validação e teste" width="80%"><br>
  <em><b>Figura 5.</b> Divisão do <i>dataset</i> em treino, validação e teste.</em>
</p>

- Sem separar dados, não há como verificar se o modelo generaliza: se as rodinhas nunca saem, como saber se a criança aprendeu?
- Primeiro, reservamos o **teste** (20%), usado uma única vez, no fim.
- O restante, `full_train`, é dividido em **treino** (60%) e **validação** (20%), onde fazemos as escolhas do modelo.
- Feitas as escolhas, retreinamos com todo o `full_train`, aproveitando também os pacientes da validação.

In [ ]:
from sklearn.model_selection import train_test_split

X_full_train, X_test, y_full_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_full_train, y_full_train, test_size=0.25, random_state=SEED)

print(f"Treino: {len(X_train)} | Validação: {len(X_val)} | Teste: {len(X_test)}")

- `random_state=SEED`: mesmo sorteio para todos.
- A ordem entre linhas de `X` e valores de `y` é preservada: cada paciente continua com a sua resposta.

## **1.6 Regressão linear simples (uma *feature*: IMC)**

Com uma só característica, a soma ponderada fica com um termo e vira a equação de uma reta:

$$\hat{y} = w_0 + w_1 \cdot \text{IMC}$$

- $w_1$: inclinação, quanto a previsão muda a cada unidade de IMC.
- $w_0$: intercepto, onde a reta cruza o eixo vertical.
- Treinar é escolher $w_0$ e $w_1$: o velho ajuste de reta, como na curva padrão do laboratório.

In [ ]:
# Colchetes duplos: o Scikit-Learn espera uma tabela, mesmo com uma só feature
X_train_bmi = X_train[["bmi"]]
X_val_bmi = X_val[["bmi"]]

X_train_bmi.shape, X_val_bmi.shape

### ***Baseline*: prever sempre a média do treino**

In [ ]:
from sklearn.dummy import DummyRegressor

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train_bmi, y_train)
baseline_pred = baseline.predict(X_val_bmi)

baseline_pred[:5]

- O `DummyRegressor` ignora o IMC e prevê sempre a média do treino: uma reta horizontal.
- Padrão do Scikit-Learn: `fit()` aprende a partir dos dados (aqui, só a média) e `predict()` devolve previsões.
- O modelo com IMC só vale a pena se errar menos que o *baseline*.

### **Ajustando a reta**

- `fit(X, y)` escolhe $w_0$ e $w_1$ pelo **método dos mínimos quadrados**: a reta com a menor soma dos quadrados das distâncias verticais até os pontos de treino.
- O que é aprendido fica guardado no modelo, em atributos terminados em `_`: `intercept_` ($w_0$) e `coef_` (um peso por *feature*; aqui, só $w_1$, por isso o `[0]`).

In [ ]:
from sklearn.linear_model import LinearRegression

simple_model = LinearRegression()
simple_model.fit(X_train_bmi, y_train)

w0 = simple_model.intercept_
w1 = simple_model.coef_[0]
print(f"Intercepto (w0): {w0:.2f}")
print(f"Inclinação (w1): {w1:.2f}")

In [ ]:
x_reta = np.linspace(X_train["bmi"].min(), X_train["bmi"].max(), 2)

fig, ax = plt.subplots()
ax.scatter(X_train["bmi"], y_train, s=18, color=CORES["azul"], label="Pacientes de treino", **ESTILO_PONTOS)
ax.plot(x_reta, w0 + w1 * x_reta, color=CORES["ocre"], label=f"Reta: ŷ = {w0:.1f} + {w1:.2f}·IMC")
ax.axhline(y_train.mean(), color=CORES["texto_secundario"], linestyle="--", label="Baseline")
ax.set(xlabel="Índice de massa corporal", ylabel="Progressão da doença")
ax.legend(loc="upper left", fontsize=9)
fig.tight_layout()
plt.show()

$$\hat{y} = -134{,}5 + 10{,}73 \cdot \text{IMC}$$

- Cada unidade a mais de IMC soma cerca de 10,7 à previsão.
- IMC = 0 não existe: $w_0$ só posiciona a reta. Fora da faixa de IMC do treino (18,9 a 41,3), a reta não tem respaldo nos dados.

### **Previsões e resíduos na validação**

$$e_i = y_i - \hat{y}_i$$

- Resíduo positivo: o modelo previu menos que o observado; negativo: previu mais.
- `predict()` aplica a reta aprendida no treino aos pacientes de validação.

In [ ]:
simple_pred = simple_model.predict(X_val_bmi)
residuos = y_val - simple_pred      # e_i = y_i - ŷ_i

x_reta = np.linspace(X_val["bmi"].min(), X_val["bmi"].max(), 2)
fig, ax = plt.subplots()
ax.vlines(X_val["bmi"], simple_pred, y_val, color=CORES["azul_acinzentado"], linewidth=1, alpha=0.6, label="Resíduo")
ax.scatter(X_val["bmi"], y_val, s=18, color=CORES["azul"], label="Pacientes de validação", **ESTILO_PONTOS)
ax.plot(x_reta, w0 + w1 * x_reta, color=CORES["ocre"], label="Reta ajustada no treino")
destaque = [87, 359]  # IMC parecido, resíduos opostos (índices válidos para SEED = 42)
ax.scatter(X_val.loc[destaque, "bmi"], y_val.loc[destaque], s=70, color=CORES["ocre"], zorder=4,
           label="Pacientes 87 e 359")
for i in destaque:
    ax.annotate(f"e = {residuos.loc[i]:.1f}", (X_val.loc[i, "bmi"], y_val.loc[i]), xytext=(8, 0),
                textcoords="offset points", va="center",
                bbox={"boxstyle": "round", "facecolor": CORES["papel"], "edgecolor": CORES["linha"]})
ax.set(xlabel="Índice de massa corporal", ylabel="Progressão da doença")
ax.legend(loc="upper left", fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
pd.DataFrame({
    "IMC": X_val.loc[destaque, "bmi"],
    "Observado": y_val.loc[destaque],
    "Previsão baseline": baseline_pred[0],
    "Resíduo baseline": y_val.loc[destaque] - baseline_pred[0],
    "Previsão reta": simple_model.predict(X_val_bmi.loc[destaque]),
    "Resíduo reta": residuos.loc[destaque],
}).round(2)

- IMC quase igual, previsões quase iguais (~142 e ~154); observados 134 e 311.
- Para um paciente, a reta erra por 8; para o outro, por 157. O IMC não carrega a informação que distingue os dois.

### **RMSE**

$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n} e_i^2}$$

- Quadrado: erros positivos e negativos não se cancelam, e erros grandes pesam mais.
- Média e raiz: o resultado volta à unidade da progressão; é o erro típico de uma previsão.

In [ ]:
from sklearn.metrics import root_mean_squared_error

baseline_rmse = root_mean_squared_error(y_val, baseline_pred)
simple_rmse = root_mean_squared_error(y_val, simple_pred)

print(f"RMSE do baseline: {baseline_rmse:.2f}")
print(f"RMSE da regressão simples: {simple_rmse:.2f}")

- O IMC reduz o RMSE de 75,64 para 58,57; ainda assim, são ~60 unidades num *target* que vai de 25 a 346.

### **Análise dos resíduos**

In [ ]:
fig, ax = plt.subplots()
ax.scatter(simple_pred, residuos, s=18, color=CORES["azul"], **ESTILO_PONTOS)
ax.axhline(0, color=CORES["texto_secundario"], linestyle="--", label="Resíduo = 0")
ax.axhline(residuos.mean(), color=CORES["ocre"], label=f"Média dos resíduos = {residuos.mean():.2f}")
ax.set(xlabel="Progressão prevista", ylabel="Resíduo")
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

residuos_train = y_train - simple_model.predict(X_train_bmi)
print(f"Média dos resíduos no treino: {residuos_train.mean():.2e}")
print(f"Média dos resíduos na validação: {residuos.mean():.2f}")

- Sem padrão evidente: os erros se espalham dos dois lados do zero.
- No treino, a média dos resíduos é zero: a reta se equilibra no centro de massa dos pontos.
- Na validação, a média é 13,55: o grupo sorteado progrediu mais do que a reta esperava. Uma única divisão é uma única fotografia.

## 1.7 Regressão linear múltipla (todas as *features*)

$$\hat{y} = w_0 + w_1\,\text{age} + w_2\,\text{sex} + w_3\,\text{bmi} + \dots + w_{10}\,\text{s6}$$

- A mesma soma ponderada, agora com 10 pesos: a reta vira um hiperplano.
- No código, só trocamos `X_train_bmi` por `X_train`.

In [ ]:
multiple_model = LinearRegression()
multiple_model.fit(X_train, y_train)
multiple_pred = multiple_model.predict(X_val)

print(f"Intercepto (w0): {multiple_model.intercept_:.2f}")

# Um peso por feature, além do intercepto
pd.Series(multiple_model.coef_, index=X_train.columns, name="peso").round(2)

- O tamanho de um peso depende da escala da *feature*: não compare pesos diretamente.
- *Features* correlacionadas, como `s1` e `s2`, dividem a contribuição entre si: interpretar pesos isolados exige cuidado.

**Variância explicada**

- A progressão varia muito entre pacientes. O *baseline* não explica nada dessa variação: todo desvio em torno da média vira erro.
- Um modelo acompanha parte da variação, e essa parte sai dos resíduos.
- **Variância explicada** = fração da variação que deixou de ser erro em relação ao *baseline*:
  1 − (soma dos resíduos² do modelo) ÷ (soma dos resíduos² do *baseline*).
- Só com o IMC: o poder preditivo vem de uma única característica, e o resto vai para os resíduos. Por isso, pacientes com IMC parecido podem ter erros tão diferentes.
- Com todas as *features*: o poder preditivo vem da combinação das características, não mais de uma reta.

In [ ]:
def resumo_validacao(previsao):
    erro = y_val - previsao
    erro_baseline = y_val - baseline_pred
    return {
        "RMSE na validação": root_mean_squared_error(y_val, previsao),
        "Variância explicada": 1 - (erro**2).sum() / (erro_baseline**2).sum(),
        "Média dos resíduos": erro.mean(),
    }


pd.DataFrame({
    "Baseline": resumo_validacao(baseline_pred),
    "Regressão simples": resumo_validacao(simple_pred),
    "Regressão múltipla": resumo_validacao(multiple_pred),
}).T.round(2)

In [ ]:
previsao_multipla = multiple_model.predict(X_val.loc[destaque])
pd.DataFrame({
    "Observado": y_val.loc[destaque],
    "Resíduo reta": residuos.loc[destaque],
    "Previsão múltipla": previsao_multipla,
    "Resíduo múltipla": y_val.loc[destaque] - previsao_multipla,
}).round(2)

- Variância explicada sobe de ~40% (IMC) para ~54% (todas as *features*).
- Paciente 359: o resíduo cai de 157 para 137. A combinação ajuda, mas ~46% da variação segue sem explicação.

In [ ]:
lims = [y_val.min(), y_val.max()]
fig, ax = plt.subplots()
ax.scatter(y_val, multiple_pred, s=22, color=CORES["verde"], label="Pacientes de validação", **ESTILO_PONTOS)
ax.plot(lims, lims, color=CORES["texto_secundario"], linestyle="--", label="Previsão = observado")
ax.set(xlabel="Progressão observada", ylabel="Progressão prevista")
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

- Pontos sobre a diagonal: previsão igual ao observado.
- Pacientes com progressão alta tendem a ficar abaixo dela: o modelo os subestima.

## **1.8 Validação cruzada**

- Uma única divisão é uma única fotografia: e se o sorteio colocou na validação pacientes "fáceis" (ou difíceis)?
- Teste: repetir a divisão treino/validação com outras sementes.

In [ ]:
# Uma única divisão é uma única fotografia: outras sementes, outros resultados
linhas = []
for semente in range(6):
    Xa, Xb, ya, yb = train_test_split(X_full_train, y_full_train, test_size=0.25, random_state=semente)
    modelo = LinearRegression().fit(Xa[["bmi"]], ya)
    erro = yb - modelo.predict(Xb[["bmi"]])
    linhas.append({"Semente": semente, "Média dos resíduos": erro.mean(), "RMSE": np.sqrt((erro**2).mean())})

pd.DataFrame(linhas).round(1)

- Mesma regressão simples, outras sementes: média dos resíduos de −6,5 a +10,3; RMSE de 58,1 a 67,5.
- O desempenho medido depende de quem caiu na validação.

In [ ]:
# Esquema da validação cruzada com 5 dobras
n_dobras = 5
fig, ax = plt.subplots(figsize=(8, 3.2))
for linha in range(n_dobras):
    for bloco in range(n_dobras):
        validacao = bloco == linha
        ax.add_patch(plt.Rectangle(
            (bloco, n_dobras - 1 - linha), 0.94, 0.8,
            facecolor=CORES["ocre"] if validacao else CORES["preenchimento_azul"],
            edgecolor=CORES["ocre"] if validacao else CORES["azul"], linewidth=1.2,
        ))
        if validacao:
            ax.text(bloco + 0.47, n_dobras - 1 - linha + 0.4, "Validação", ha="center", va="center",
                    color=CORES["papel"], fontsize=10)
    ax.text(-0.15, n_dobras - 1 - linha + 0.4, f"Dobra {linha + 1}", ha="right", va="center")
ax.set(xlim=(-1.3, n_dobras), ylim=(-0.3, n_dobras))
ax.set_title("full_train dividido em 5 partes", loc="center")
ax.axis("off")
ax.legend(handles=[
    plt.Rectangle((0, 0), 1, 1, facecolor=CORES["preenchimento_azul"], edgecolor=CORES["azul"], label="Treino"),
    plt.Rectangle((0, 0), 1, 1, facecolor=CORES["ocre"], label="Validação"),
], loc="upper center", bbox_to_anchor=(0.5, 0), ncol=2, frameon=False)
fig.tight_layout()
plt.show()

- O `full_train` é dividido em 5 partes; cada uma serve **uma vez** de validação, enquanto as outras 4 treinam.
- Todo paciente valida exatamente uma vez: nenhum resultado depende de uma divisão "fácil".
- O resultado são 5 RMSEs, resumidos por estatísticas: média (valor típico) e desvio padrão (dispersão).
- A validação cruzada substitui a divisão fixa treino/validação, que usamos para aprender o passo a passo.

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

modelos_cv = {
    "Baseline": (DummyRegressor(strategy="mean"), X_full_train),
    "Regressão simples": (LinearRegression(), X_full_train[["bmi"]]),
    "Regressão múltipla": (LinearRegression(), X_full_train),
}

# O Scikit-Learn devolve o RMSE com sinal negativo (convenção "maior é melhor"); invertemos o sinal
rmse_cv = {
    nome: -cross_val_score(modelo, X_cv, y_full_train, cv=kf, scoring="neg_root_mean_squared_error")
    for nome, (modelo, X_cv) in modelos_cv.items()
}

pd.DataFrame({nome: {"Média": s.mean(), "Desvio padrão": s.std()} for nome, s in rmse_cv.items()}).T.round(2)

- `neg_root_mean_squared_error`: por convenção do Scikit-Learn, "maior é melhor"; por isso o sinal negativo, que invertemos.
- *Baseline* 78,1 ± 5,6; simples 62,5 ± 1,7; múltipla 55,4 ± 2,4.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
for posicao, (nome, valores) in enumerate(rmse_cv.items()):
    ax.scatter(valores, [posicao] * len(valores), s=40, color=CORES["azul"], **ESTILO_PONTOS)
    ax.scatter(valores.mean(), posicao, marker="x", s=140, linewidths=3, color=CORES["ocre"], zorder=3)
ax.set_yticks(range(len(rmse_cv)), list(rmse_cv))
ax.grid(False)
ax.grid(axis="x")
ax.set(xlabel="RMSE na validação cruzada (5 dobras; x = média)")
fig.tight_layout()
plt.show()

- A ordem dos modelos se mantém em todas as dobras: a regressão múltipla é a escolha, agora com respaldo de um desempenho médio.

## **1.9 Avaliação final no conjunto de teste**

- Modelo escolhido (regressão múltipla) retreinado com todo o `full_train`.
- O teste, reservado desde o início, é usado **uma única vez**: a avaliação em pacientes que não participaram de nenhuma escolha.

In [ ]:
# Modelo escolhido, retreinado com todo o full_train, avaliado uma única vez no teste
final_model = LinearRegression().fit(X_full_train, y_full_train)
final_pred = final_model.predict(X_test)

baseline_final = DummyRegressor(strategy="mean").fit(X_full_train, y_full_train)

print(f"RMSE no teste (regressão múltipla): {root_mean_squared_error(y_test, final_pred):.2f}")
print(f"RMSE no teste (baseline): {root_mean_squared_error(y_test, baseline_final.predict(X_test)):.2f}")

- RMSE no teste: 53,85, contra 73,22 do *baseline*.
- Dentro da faixa da validação cruzada (55,4 ± 2,4): a avaliação se confirma em pacientes novos.

In [ ]:
lims = [y_test.min(), y_test.max()]
fig, ax = plt.subplots()
ax.scatter(y_test, final_pred, s=22, color=CORES["verde"], label="Pacientes de teste", **ESTILO_PONTOS)
ax.plot(lims, lims, color=CORES["texto_secundario"], linestyle="--", label="Previsão = observado")
ax.set(xlabel="Progressão observada", ylabel="Progressão prevista")
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

**De volta à pergunta central:** o modelo prevê a progressão do diabetes melhor que o *baseline*, mas com erro típico de ~54 unidades e cerca de metade da variação sem explicação. As características registradas carregam parte da informação, mas não toda.

# **Parte 2 - Classificação: nódulos de mama**

## **2.1 Problema biológico**

> **Pergunta central:** as medidas dos núcleos celulares distinguem nódulos benignos de malignos?

<p align="center">
  <img src="https://raw.githubusercontent.com/lucasofalonso/workshop-ml-biologia/main/images/dataset_cancer.png" alt="Dos nódulos aos dados" width="75%"><br>
  <em><b>Figura 6.</b> Ilustração do <i>dataset</i> Breast Cancer Wisconsin.</em>
</p>

- O *dataset*, ilustrado na Figura 6, reúne **569 nódulos de mama**. De cada um, uma amostra obtida por punção com agulha fina foi fotografada ao microscópio, e foram medidas características dos núcleos celulares, como raio, textura e concavidade.
- Cada característica aparece em três versões: média (`mean`), erro padrão (`error`) e pior valor (`worst`), o que totaliza **30 *features***.

Para responder a essa pergunta, um regressor não é adequado, porque prevê valores numéricos. Precisamos de outro tipo de modelo supervisionado: os **classificadores**, que preveem **classes**.

**Depois do treinamento e da validação, dada a amostra de um nódulo, o modelo a atribui à classe benigna ou maligna.**

- Em perguntas de "sim" ou "não", com duas classes possíveis, o *target* costuma ser codificado como 0 e 1. A escolha de qual classe é 0 é uma convenção; o importante é manter a consistência.
- Neste *dataset*, 0 = maligno e 1 = benigno. Para evitar confusão, vamos trocar os códigos pelos nomes das classes.
- Usaremos maligno = 1 (classe positiva, a de interesse clínico) e benigno = 0.

## **2.2 Carregando os dados**

In [ ]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer(as_frame=True)

X_bc = cancer.data
# Nomes das classes, usados nos gráficos da exploração
diagnostico = cancer.target.map({0: "maligno", 1: "benigno"}).rename("diagnostico")

# Target binário usado nos modelos: 1 = maligno (classe positiva), 0 = benigno
y_bc = (diagnostico == "maligno").astype(int).rename("maligno")

X_bc.shape

## **2.3 Exploração inicial**

In [ ]:
X_bc.head()

In [ ]:
print(f"Valores ausentes: {X_bc.isna().sum().sum()}")
diagnostico.value_counts().to_frame("nódulos").assign(proporcao=lambda t: (t["nódulos"] / t["nódulos"].sum()).round(3))

- 30 *features* numéricas, sem valores ausentes.
- 357 benignos (63%) e 212 malignos (37%).

### **Correlação com um *target* categórico**

- Na Parte 1, o *target* era um número, e o *r* media se *feature* e progressão cresciam juntas.
- Agora o *target* é uma classe. Codificando maligno = 1 e benigno = 0, a mesma correlação de Pearson passa a medir **o quanto os valores de uma *feature* mudam de uma classe para a outra**.
- A nuvem de pontos vira duas colunas, uma por classe; a "reta de tendência" liga as médias das duas colunas.

In [ ]:
# Diagnóstico codificado: maligno = 1, benigno = 0
corr_diagnostico = X_bc.corrwith(y_bc).sort_values()

# Ponte com a Parte 1: uma feature forte e uma fraca, com o diagnóstico no eixo horizontal
exemplos_ponte = ["worst concave points", "symmetry error"]
espalhamento = np.random.default_rng(SEED).uniform(-0.12, 0.12, len(y_bc))
cores_pontos = np.where(y_bc == 1, CORES["ocre"], CORES["azul"])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, feature in zip(axes, exemplos_ponte):
    ax.scatter(y_bc + espalhamento, X_bc[feature], s=12, c=cores_pontos, alpha=0.5, linewidths=0)
    medias = X_bc[feature].groupby(y_bc).mean()
    ax.plot([0, 1], medias.values, color=CORES["texto_secundario"], marker="x", markersize=12,
            markeredgewidth=3, label="Média de cada classe")
    ax.set(title=f"{feature}\nr = {corr_diagnostico[feature]:.2f}", xticks=[0, 1],
           xticklabels=["benigno (0)", "maligno (1)"], ylabel=feature)
    ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

- Esquerda: médias distantes e colunas pouco sobrepostas; *r* alto.
- Direita: médias quase iguais e colunas sobrepostas; *r* perto de zero.
- *r* alto significa que conhecer o valor da *feature* ajuda a adivinhar a classe.

<a name="poder-discriminativo"></a>
### **Correlação de cada *feature* com o diagnóstico: poder discriminativo**

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.barh(corr_diagnostico.index, corr_diagnostico.values,
        color=np.where(corr_diagnostico.values > 0, CORES["ocre"], CORES["azul"]))
ax.axvline(0, color=CORES["texto_secundario"], linewidth=0.8)
ax.tick_params(axis="y", labelsize=8)
ax.grid(False)
ax.grid(axis="x")
ax.set(title="Correlação de cada feature com o diagnóstico", xlabel="r de Pearson (maligno = 1)")
fig.tight_layout()
plt.show()

- *r* positivo: valores maiores nos malignos.
- No topo, medidas de tamanho e de irregularidade do núcleo (raio, perímetro, pontos côncavos).

In [ ]:
# As 4 features mais e as 4 menos correlacionadas com o diagnóstico
ordem = corr_diagnostico.abs().sort_values(ascending=False).index
selecionadas = list(ordem[:4]) + list(ordem[-4:])
cores_diagnostico = {"benigno": CORES["azul"], "maligno": CORES["ocre"]}

fig, axes = plt.subplots(2, 4, figsize=(13, 7))
for ax, feature in zip(axes.flat, selecionadas):
    sns.boxplot(x=diagnostico, y=X_bc[feature], order=["benigno", "maligno"], width=0.5, ax=ax, **ESTILO_BOXPLOT)
    sns.stripplot(x=diagnostico, y=X_bc[feature], order=["benigno", "maligno"], hue=diagnostico, palette=cores_diagnostico,
                  legend=False, size=2.5, alpha=0.5, ax=ax)
    ax.set(title=f"{feature}\nr = {corr_diagnostico[feature]:.2f}", xlabel="", ylabel="")
axes[0, 0].set_ylabel("Mais correlacionadas")
axes[1, 0].set_ylabel("Menos correlacionadas")
fig.tight_layout()
plt.show()

- O *boxplot* resume as duas colunas de pontos: cada painel mostra o **poder discriminativo** de uma característica morfológica.
- Linha de cima: caixas distantes, pouca sobreposição; a *feature* sozinha já separa bem as classes.
- Linha de baixo: caixas sobrepostas; a *feature* sozinha quase não distingue benignos de malignos.
- Essa leitura dos *boxplots* volta na Parte 3, com as proteínas dos camundongos.

### **Correlação entre *features*: redundância**

- Outra pergunta: duas *features* contam a mesma história?
- Esta análise **não usa o diagnóstico**: só as *features* entre si.

In [ ]:
# As 10 features de média: matriz legível com anotações
colunas_media = [col for col in X_bc.columns if col.startswith("mean")]
corr_features = X_bc[colunas_media].corr()
rotulos = [col.replace("mean ", "") for col in colunas_media]

fig, ax = plt.subplots(figsize=(8.5, 7))
sns.heatmap(corr_features, annot=True, fmt=".2f", annot_kws={"size": 8}, cmap=CMAP_CORRELACAO,
            vmin=-1, vmax=1, xticklabels=rotulos, yticklabels=rotulos, ax=ax)
ax.set_title("Correlação entre as features de média")
fig.tight_layout()
plt.show()

- Raio, perímetro e área: *r* ≈ 0,99. Pela geometria, são quase a mesma medida: **redundantes**.
- Concavidade e pontos côncavos: *r* = 0,92, outro grupo que conta a mesma história.
- Se uma *feature* já separa as classes, uma redundante a ela acrescenta pouca informação nova.
- **Com o diagnóstico:** qual *feature* separa as classes? **Entre *features*:** quais *features* são redundantes? A segunda pergunta não precisa de rótulos e volta na Parte 3, quando agruparmos proteínas redundantes.

## **2.4 Divisão estratificada do *dataset***

- Classes desbalanceadas: 63% benignos e 37% malignos.
- `stratify` preserva essa proporção no treino, na validação e no teste.

In [ ]:
X_bc_full_train, X_bc_test, y_bc_full_train, y_bc_test = train_test_split(
    X_bc, y_bc, test_size=0.2, stratify=y_bc, random_state=SEED
)
X_bc_train, X_bc_val, y_bc_train, y_bc_val = train_test_split(
    X_bc_full_train, y_bc_full_train, test_size=0.25, stratify=y_bc_full_train, random_state=SEED
)

pd.DataFrame({
    nome: partes.value_counts(normalize=True)
    for nome, partes in [("Completo", y_bc), ("Treino", y_bc_train), ("Validação", y_bc_val), ("Teste", y_bc_test)]
}).rename(index={0: "benigno", 1: "maligno"}).round(3)

**Acurácia:** fração das previsões que estão corretas, seja qual for a classe.

- Com classes desbalanceadas, a acurácia engana: um modelo que sempre responde "benigno" acerta 63% das vezes sem aprender nada.

## **2.5 *Baseline*: prever sempre a classe mais frequente**

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

baseline_bc = DummyClassifier(strategy="most_frequent")
baseline_bc.fit(X_bc_train, y_bc_train)
baseline_bc_pred = baseline_bc.predict(X_bc_val)

print(f"Acurácia do baseline: {accuracy_score(y_bc_val, baseline_bc_pred):.3f}")

- O `DummyClassifier` sempre prevê a classe mais frequente, "benigno".
- Acurácia de 62%, igual à proporção de benignos na validação, e nenhum maligno detectado.
- É a referência mínima: um classificador útil precisa superá-la, especialmente nos malignos.

## **2.6 Regressão logística**

- Apesar do nome, é um **classificador**.
- Assim como a regressão linear, combina as *features* em uma soma ponderada; depois, transforma o resultado em uma **probabilidade** entre 0 e 1.
- Acima de 0,5, prevê maligno (1); abaixo, benigno (0).

<p align="center">
  <img src="https://raw.githubusercontent.com/lucasofalonso/workshop-ml-biologia/main/images/classificacao_lr.png" alt="Da probabilidade à classe" width="75%"><br>
  <em><b>Figura 7.</b> Da probabilidade estimada à classe prevista, com limiar de 50%.</em>
</p>

- Probabilidades próximas, como 49% e 51%, podem virar classes diferentes: a probabilidade carrega mais informação do que a classe.
- A regressão logística é sensível à escala, então padronizamos as *features* com `StandardScaler()`.
- **`Pipeline`:** junta etapas em um único objeto, cada uma com um nome. `fit` ajusta todas as etapas no treino, e `predict` reaproveita esses ajustes nos dados novos. Por isso, a média e o desvio da padronização vêm só do treino.

<p align="center">
  <img src="https://raw.githubusercontent.com/lucasofalonso/workshop-ml-biologia/main/images/pipeline_funcionamento.png" alt="Pipeline: etapas conectadas" width="75%"><br>
  <em><b>Figura 8.</b> Funcionamento de um <i>pipeline</i>. No nosso, há só duas etapas: padronização e modelo.</em>
</p>

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Padronização + regressão logística em um único objeto (pipeline)
logistic = Pipeline([
    ("padronizacao", StandardScaler()),
    ("modelo", LogisticRegression(max_iter=1000)),
])
logistic.fit(X_bc_train, y_bc_train)
logistic_pred = logistic.predict(X_bc_val)

In [ ]:
# A regressão logística devolve uma probabilidade para cada classe; predict() usa o limiar de 0,5
probabilidades = pd.DataFrame(logistic.predict_proba(X_bc_val), columns=["P(benigno)", "P(maligno)"]).round(3)
probabilidades.assign(previsto=logistic_pred, observado=y_bc_val.values).head()

- `predict_proba()`: probabilidade de cada classe; `predict()`: a classe com probabilidade maior que 0,5.

## **2.7 Métricas de performance para um classificador**

Vamos exemplificar algumas métricas e performance relevantes para classificadores, particularizando para o modelo de regressão logística. Posteriormente utilizaremos as mesmas métricas nas árvores de decisão.

**Matriz de confusão:** linhas = classe observada; colunas = classe prevista. A diagonal reúne os acertos.

- **Acurácia:** fração das previsões corretas.
- **Precisão** : quando o modelo prediz uma classe, quantas vezes acerta.
- ***Recall*** : Entre os exemplos reais de uma classe, quantos o modelo detecta/recupera.
- **F1:** equilíbrio entre precisão e *recall*.
- `support`: número de nódulos de cada classe.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report


def matriz_confusao(observado, previsto, titulo):
    fig, ax = plt.subplots(figsize=(4.2, 3.8))
    ConfusionMatrixDisplay.from_predictions(
        observado, previsto, labels=[1, 0], display_labels=["maligno", "benigno"], cmap=CMAP_CONTAGEM, colorbar=False, ax=ax
    )
    ax.grid(False)
    ax.set(title=titulo, xlabel="Previsto", ylabel="Observado")
    fig.tight_layout()
    plt.show()


print(f"Acurácia: {accuracy_score(y_bc_val, logistic_pred):.3f}")
matriz_confusao(y_bc_val, logistic_pred, "Regressão logística (validação)")
print(classification_report(y_bc_val, logistic_pred, digits=3, target_names=["benigno", "maligno"]))

- Acurácia de 97%, contra 62% do *baseline*.
- 42 de 43 malignos identificados (*recall* ≈ 0,98); 2 benignos classificados como malignos.
- Em diagnóstico, o erro mais grave é o maligno classificado como benigno: por isso o *recall* dos malignos merece atenção especial.

## **2.8 Árvores de decisão**

### **Uma única árvore de decisão**

- Diferente da regressão logística, a árvore não é sensível à escala: não precisamos padronizar.
- A árvore faz perguntas do tipo "esta *feature* é menor ou igual a este valor?" e divide os nódulos em dois grupos.
- Em cada passo, o algoritmo testa todas as *features* e todos os limiares possíveis e escolhe a divisão que melhor separa as classes, deixando os grupos filhos mais "puros".
- Assim, a árvore decide gradativamente quais *features* usar e em que ordem.
- `max_depth=3`: no máximo três níveis de perguntas, para a árvore não decorar os dados de treino.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

tree = DecisionTreeClassifier(max_depth=3, random_state=SEED)
tree.fit(X_bc_train, y_bc_train)
tree_pred = tree.predict(X_bc_val)

print(f"Acurácia: {accuracy_score(y_bc_val, tree_pred):.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6.5))
plot_tree(tree, feature_names=X_bc.columns, class_names=["benigno", "maligno"], filled=False,
          impurity=False, proportion=True, rounded=True, fontsize=10, ax=ax)
fig.tight_layout()
plt.show()

**Como ler a árvore**

- **Raiz:** 100% dos nódulos de treino; `value` = fração de [benigno, maligno] no nó.
- Cada nó é classificado por **voto de maioria**: a classe mais frequente entre seus nódulos.
- Raiz: `worst perimeter <= 112.8`. À direita, 34,9% dos nódulos, com [0,067; 0,933]: maioria maligna. À esquerda, maioria benigna.
- **Folhas:** a classificação final. As folhas malignas reúnem 32,0% + 3,2% + 0,9% + 0,6% = 36,7% dos nódulos de treino; o restante vai para benigno.
- Uma folha pode conter nódulos da outra classe: são os erros da árvore no próprio treino.

In [ ]:
matriz_confusao(y_bc_val, tree_pred, "Árvore de decisão (validação)")

# Quanto cada feature contribuiu para as divisões da árvore
importancias = pd.Series(tree.feature_importances_, index=X_bc.columns, name="importância")
importancias[importancias > 0].sort_values(ascending=False).round(3)

- **Importância de uma *feature***: o quanto ela contribuiu, somando todas as divisões em que aparece, para separar as classes.
- Só as *features* usadas nas divisões têm importância maior que zero.

### **Random forest: um conjunto de árvores de decisão**

- Uma única árvore é instável: pequenas mudanças no treino podem mudar as perguntas e o resultado.
- A *random forest* treina muitas árvores diferentes e combina suas respostas (Figura 9).
- Duas fontes de diversidade:
  - cada árvore recebe um sorteio dos nódulos de treino **com reposição**: alguns se repetem, outros ficam de fora;
  - em cada nó, só um sorteio das *features* concorre à melhor divisão.
- **Previsão:** cada árvore vota. A intuição é o **voto de maioria**; o Scikit-Learn calcula a média das probabilidades das árvores.

<p align="center">
  <img src="https://raw.githubusercontent.com/lucasofalonso/workshop-ml-biologia/main/images/random_forest.png" alt="Floresta aleatória" width="75%"><br>
  <em><b>Figura 9.</b> <i>Random forest</i>: árvores treinadas com sorteios diferentes e uma decisão coletiva.</em>
</p>

- `n_estimators=100`: número de árvores.
- Sem `max_depth`, cada árvore cresce até decorar o próprio treino; a combinação das 100 compensa isso.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 100 árvores; cada uma treinada com um sorteio dos nódulos de treino
rf = RandomForestClassifier(n_estimators=100, random_state=SEED)
rf.fit(X_bc_train, y_bc_train)
rf_pred = rf.predict(X_bc_val)

print(f"Acurácia: {accuracy_score(y_bc_val, rf_pred):.3f}")
matriz_confusao(y_bc_val, rf_pred, "Random forest (validação)")

- Acurácia de 97%, contra 92% da árvore única.
- 41 de 43 malignos detectados, contra 38 da árvore; desempenho próximo ao da regressão logística (42 de 43). A escolha entre os modelos fica para a validação cruzada.

In [ ]:
# Voto de cada árvore para um nódulo da validação
exemplo = 171
nodulo = X_bc_val.loc[[exemplo]]

# As árvores internas foram treinadas sem nomes de colunas: passamos o array
votos = pd.Series([int(arvore.predict(nodulo.to_numpy())[0]) for arvore in rf.estimators_])
print(f"Votos: {(votos == 1).sum()} maligno, {(votos == 0).sum()} benigno")
print(f"Probabilidade média de maligno: {rf.predict_proba(nodulo)[0, 1]:.2f}")
print(f"Previsto: {'maligno' if rf.predict(nodulo)[0] == 1 else 'benigno'}; "
      f"observado: {'maligno' if y_bc_val[exemplo] == 1 else 'benigno'}")

- Nódulo 171: 56 árvores votam maligno e 44 votam benigno. A floresta prevê maligno, e ele é de fato maligno.
- Como cada árvore cresce até folhas puras, a probabilidade dada por cada uma é 0 ou 1: a média das probabilidades coincide com a fração de votos.

In [ ]:
# Importância das features: árvore única vs. floresta
pd.DataFrame({
    "árvore": tree.feature_importances_,
    "random forest": rf.feature_importances_,
}, index=X_bc.columns).sort_values("random forest", ascending=False).head(10).round(3)

- A árvore única concentra a importância em 6 *features*; só `worst perimeter` responde por 77%.
- A floresta distribui a importância entre as 30, com destaque para concavidade e tamanho: as mesmas de maior [poder discriminativo](#poder-discriminativo).

## **2.9 Seleção do melhor modelo: validação cruzada e GridSearch**

- Cada modelo tem **hiperparâmetros**: escolhas feitas antes do treino, como a profundidade da árvore.
- **Busca em grade (*GridSearch*)**: testa todas as combinações de uma lista de valores e fica com a melhor.
- Cada combinação é avaliada por **validação cruzada estratificada**: 5 dobras, cada uma com a mesma proporção de benignos e malignos.
- Tudo acontece dentro do `full_train`; o teste continua guardado.
- Hiperparâmetros testados:
  - Regressão logística, `C`: o quanto o modelo pode se ajustar ao treino. Com `C` pequeno, o modelo fica mais contido.
  - Árvore de decisão, `max_depth`: número máximo de níveis de perguntas.
  - *Random forest*, `max_depth` e `max_features`: profundidade das árvores e fração de *features* sorteada em cada nó.

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, ParameterGrid, StratifiedKFold

pipelines = {
    "Baseline": Pipeline([
        ("modelo", DummyClassifier(strategy="most_frequent")),
    ]),
    "Regressão logística": Pipeline([
        ("padronizacao", StandardScaler()),
        ("modelo", LogisticRegression(max_iter=1000)),
    ]),
    "Árvore de decisão": Pipeline([
        ("modelo", DecisionTreeClassifier(random_state=SEED)),
    ]),
    "Random forest": Pipeline([
        ("modelo", RandomForestClassifier(n_estimators=100, random_state=SEED)),
    ]),
}

param_grids = {
    "Baseline": {},
    "Regressão logística": {"modelo__C": [0.01, 0.1, 1, 10, 100]},
    "Árvore de decisão": {"modelo__max_depth": [2, 3, 4, 5, None]},
    "Random forest": {
        "modelo__max_depth": [3, 5, None],
        "modelo__max_features": ["sqrt", 0.3],
    },
}

# Conferência: toda combinação é aceita pelo pipeline correspondente
for nome, grid in param_grids.items():
    for params in ParameterGrid(grid):
        clone(pipelines[nome]).set_params(**params)

pd.DataFrame({
    "combinações": {nome: len(ParameterGrid(grid)) for nome, grid in param_grids.items()},
}).assign(**{"modelos treinados (× 5 dobras)": lambda df: df["combinações"] * 5})

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
metricas = {"acurácia": "accuracy", "recall (maligno)": "recall"}

# Para cada modelo: busca em grade com validação cruzada estratificada.
# refit: a melhor combinação, pelo recall dos malignos, é retreinada com todo o full_train.
buscas = {}
for nome, pipeline in pipelines.items():
    buscas[nome] = GridSearchCV(
        pipeline, param_grids[nome], cv=skf, scoring=metricas, refit="recall (maligno)", n_jobs=-1
    ).fit(X_bc_full_train, y_bc_full_train)

# Melhor combinação de cada modelo: média e desvio padrão nas 5 dobras
resultados = pd.DataFrame.from_dict({
    nome: {
        "melhores hiperparâmetros": busca.best_params_,
        **{
            f"{metrica} ({estatistica})": busca.cv_results_[f"{chave}_test_{metrica}"][busca.best_index_]
            for metrica in metricas
            for estatistica, chave in [("média", "mean"), ("dp", "std")]
        },
    }
    for nome, busca in buscas.items()
}, orient="index")

resultados.round(3)

- Critério de escolha: *recall* dos malignos, o erro mais grave em diagnóstico.
- Regressão logística (`C=1`): *recall* de 0,953 e acurácia de 0,974.
- *Random forest*: *recall* de 0,947. A diferença para a regressão logística é menor que a variação entre as dobras (dp ≈ 0,04): na prática, um empate.
- Árvore única: a melhor profundidade foi sem limite, mas ela ainda fica abaixo das outras (*recall* de 0,894).
- O *baseline* não detecta nenhum maligno: *recall* 0.

## **2.10 Avaliação final no conjunto de teste**

- O vencedor da validação cruzada já foi retreinado com todo o `full_train` (`refit`).
- O teste é usado **uma única vez**, para estimar o desempenho em nódulos nunca vistos.

In [ ]:
# Vencedor: maior recall médio dos malignos na validação cruzada
vencedor = resultados["recall (maligno) (média)"].idxmax()
final_bc = buscas[vencedor].best_estimator_
final_bc_pred = final_bc.predict(X_bc_test)

print(f"Modelo vencedor: {vencedor} {buscas[vencedor].best_params_}")
print(f"Acurácia no teste: {accuracy_score(y_bc_test, final_bc_pred):.3f}")
matriz_confusao(y_bc_test, final_bc_pred, f"{vencedor} (teste)")
print(classification_report(y_bc_test, final_bc_pred, digits=3, target_names=["benigno", "maligno"]))

- Acurácia de 96,5% no teste.
- 39 de 42 malignos detectados (*recall* de 0,929); 1 benigno classificado como maligno.
- O *recall* ficou dentro da faixa da validação cruzada (0,953 ± 0,040): o desempenho se manteve em dados novos.

**De volta à pergunta central:** sim, as medidas dos núcleos distinguem benignos de malignos, com acerto bem acima do *baseline*. Ainda assim, 3 malignos passaram como benignos, o erro que mais importa na clínica.

# **Parte 3 - Métodos não supervisionados: os nódulos sem o diagnóstico**

> **Pergunta central:** sem usar o diagnóstico, a estrutura das 30 medidas revela grupos de nódulos? Quais *features* contam a mesma história?

- Mesmo *dataset* da Parte 2, agora só com `X`: o diagnóstico **não entra em nenhum ajuste** e serve apenas para interpretar.
- É o mesmo procedimento usado em dados ômicos, numa escala em que dá para enxergar cada passo.

## **3.1 Padronização**

In [ ]:
from sklearn.decomposition import PCA

# Cada feature com média 0 e desvio padrão 1: nenhuma domina pelo tamanho dos números
padronizador = StandardScaler().set_output(transform="pandas")
X_bc_pad = padronizador.fit_transform(X_bc)

exemplos_escala = ["mean area", "mean smoothness", "worst concave points"]
pd.concat({
    "Antes": X_bc[exemplos_escala].agg(["mean", "std"]).T,
    "Depois": X_bc_pad[exemplos_escala].agg(["mean", "std"]).T,
}, axis=1).round(3) + 0.0

- Distâncias e variâncias dependem da escala: sem padronizar, a área (centenas) dominaria a suavidade (~0,1).

## **3.2 Análise de componentes principais (PCA)**

In [ ]:
pca_bc = PCA()
componentes_bc = pca_bc.fit_transform(X_bc_pad)   # fit recebe só X: nenhum rótulo

nomes_pc = [f"PC{i + 1}" for i in range(pca_bc.n_components_)]
scores_bc = pd.DataFrame(componentes_bc, columns=nomes_pc, index=X_bc.index).assign(diagnostico=diagnostico)
loadings_bc = pd.DataFrame(pca_bc.components_.T, index=X_bc.columns, columns=nomes_pc)
explicada_bc = pd.Series(pca_bc.explained_variance_ratio_, index=nomes_pc)

n = 15
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(explicada_bc.index[:n], explicada_bc.values[:n], color=CORES["azul"], label="Por componente")
ax.plot(explicada_bc.index[:n], explicada_bc.cumsum().values[:n], color=CORES["ocre"], marker="o", label="Acumulada")
ax.set(xlabel="Componente", ylabel="Fração da variância explicada", ylim=(0, 1))
ax.tick_params(axis="x", rotation=45)
ax.legend()
fig.tight_layout()
plt.show()

- PC1 resume 44% da variação das 30 medidas; PC1 e PC2 juntas, 63%.

In [ ]:
rotulo_pc1 = f"PC1 ({explicada_bc['PC1']:.0%} da variância)"
rotulo_pc2 = f"PC2 ({explicada_bc['PC2']:.0%} da variância)"

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharex=True, sharey=True)
axes[0].scatter(scores_bc["PC1"], scores_bc["PC2"], s=18, color=CORES["azul_acinzentado"], **ESTILO_PONTOS)
axes[0].set(title="O que o PCA vê (sem rótulos)", xlabel=rotulo_pc1, ylabel=rotulo_pc2)
sns.scatterplot(data=scores_bc, x="PC1", y="PC2", hue="diagnostico", palette=cores_diagnostico, s=22,
                ax=axes[1], **ESTILO_PONTOS)
axes[1].set(title="Diagnóstico, só para interpretar", xlabel=rotulo_pc1, ylabel=rotulo_pc2)
axes[1].legend(title="")
fig.tight_layout()
plt.show()

- O PCA não viu o diagnóstico, mas benignos e malignos se separam ao longo de PC1.

In [ ]:
def top_loadings_bc(pc, n=10):
    """Features com maior |loading| numa componente, mantendo o sinal."""
    return loadings_bc[pc].reindex(loadings_bc[pc].abs().sort_values(ascending=False).index)[:n]


fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for ax, pc in zip(axes, ["PC1", "PC2"]):
    top = top_loadings_bc(pc)[::-1]
    ax.barh(top.index, top.values, color=np.where(top.values > 0, CORES["azul"], CORES["ocre"]))
    ax.axvline(0, color=CORES["texto_secundario"], linewidth=0.8)
    ax.tick_params(axis="y", labelsize=9)
    ax.grid(False)
    ax.grid(axis="x")
    ax.set(title=f"{pc}: 10 maiores |loadings|", xlabel="Loading")
fig.tight_layout()
plt.show()

- *Loading*: peso de cada *feature* na construção da componente. O sinal é arbitrário; importa se as *features* apontam no mesmo sentido.
- PC1: concavidade, pontos côncavos e compacidade, as mesmas que mais separaram as classes na Parte 2.
- PC2: dimensão fractal contra tamanho (raio).

## **3.3 K-means**

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

linhas = []
for k in range(2, 9):
    grupos = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit_predict(X_bc_pad)
    linhas.append({
        "k": k,
        "silhueta": silhouette_score(X_bc_pad, grupos),
        "ARI com o diagnóstico": adjusted_rand_score(y_bc, grupos),  # só para interpretar
    })

pd.DataFrame(linhas).set_index("k").round(3)

- **Silhueta**: o quanto os grupos estão separados entre si; não usa rótulos. Máxima em k = 2.
- **ARI**: concordância entre os grupos e o diagnóstico (1 = idênticos, 0 = acaso); usado só para interpretar.

In [ ]:
kmeans_bc = KMeans(n_clusters=2, n_init=20, random_state=SEED).fit(X_bc_pad)
grupos_bc = pd.Series(kmeans_bc.labels_, index=X_bc.index, name="grupo")

pd.crosstab(grupos_bc, diagnostico)

In [ ]:
# Centroides levados para o plano do PCA
centroides_pc = pca_bc.transform(pd.DataFrame(kmeans_bc.cluster_centers_, columns=X_bc.columns))[:, :2]
cores_grupo = {0: CORES["verde"], 1: CORES["azul_acinzentado"]}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharex=True, sharey=True)
for grupo, cor in cores_grupo.items():
    no_grupo = grupos_bc == grupo
    axes[0].scatter(scores_bc.loc[no_grupo, "PC1"], scores_bc.loc[no_grupo, "PC2"], s=18, color=cor,
                    label=f"Grupo {grupo}", **ESTILO_PONTOS)
axes[0].scatter(centroides_pc[:, 0], centroides_pc[:, 1], marker="x", s=160, linewidths=3, color=CORES["ocre"],
                label="Centroide", zorder=3)
axes[0].set(title="K-means com k = 2", xlabel=rotulo_pc1, ylabel=rotulo_pc2)
axes[0].legend(fontsize=9)
sns.scatterplot(data=scores_bc, x="PC1", y="PC2", hue="diagnostico", palette=cores_diagnostico, s=22,
                ax=axes[1], **ESTILO_PONTOS)
axes[1].set(title="Diagnóstico, só para interpretar", xlabel=rotulo_pc1, ylabel=rotulo_pc2)
axes[1].legend(title="")
fig.tight_layout()
plt.show()

- Um grupo reúne quase só malignos (175 de 189); o outro, quase só benignos (343 de 380).
- 518 de 569 nódulos (91%) caem no grupo "do seu diagnóstico", sem que o K-means soubesse qual era.

## **3.4 Clusterização hierárquica das *features***

- Agora agrupamos *features*, não nódulos: a pergunta da redundância da Parte 2, respondida por um algoritmo.
- Distância entre duas *features*: 1 − |*r*|; correlação positiva ou negativa conta como parecida.
- Calculada só no `full_train`, porque o resultado vai alimentar um classificador.

In [ ]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform

corr_features_bc = X_bc_full_train.corr()
distancia_bc = 1 - corr_features_bc.abs()
arvore_bc = linkage(squareform(distancia_bc.values, checks=False), method="complete")

grade = sns.clustermap(corr_features_bc, row_linkage=arvore_bc, col_linkage=arvore_bc, cmap=CMAP_CORRELACAO,
                       vmin=-1, vmax=1, xticklabels=True, yticklabels=True, figsize=(11, 11),
                       tree_kws={"colors": CORES["texto_secundario"]}, cbar_kws={"label": "Correlação (r)"})
grade.ax_heatmap.tick_params(labelsize=8)
grade.ax_heatmap.set(xlabel="", ylabel="")
plt.show()

- O dendrograma junta, passo a passo, as *features* mais parecidas.
- Quadrados quentes ao longo da diagonal: blocos de *features* redundantes.

In [ ]:
# Corte em t = 0.2: dentro de cada bloco, todos os pares têm |r| >= 0.8
blocos_bc = pd.Series(fcluster(arvore_bc, t=0.2, criterion="distance"), index=corr_features_bc.index, name="bloco")
print(f"Blocos: {blocos_bc.nunique()}")

blocos_bc.groupby(blocos_bc).apply(lambda grupo: grupo.index.tolist()).loc[lambda s: s.str.len() > 1]

- 30 *features* viram 16 blocos.
- Raio, perímetro e área (média e pior valor) formam um único bloco: seis colunas, uma história.

In [ ]:
# Representante de cada bloco: a feature mais próxima das demais do mesmo bloco (sem rótulos)
representantes_bc = [
    distancia_bc.loc[grupo.index, grupo.index].sum(axis=1).idxmin()
    for _, grupo in blocos_bc.groupby(blocos_bc)
]
print(f"{len(representantes_bc)} representantes:")
representantes_bc

### **Classificador com as representantes**

- A mesma regressão logística da Parte 2, com 30 *features* ou só com as 16 representantes.

In [ ]:
from sklearn.model_selection import cross_validate

comparacao = {}
for nome, colunas in [("30 features", list(X_bc.columns)), ("16 representantes", representantes_bc)]:
    modelo = Pipeline([
        ("padronizacao", StandardScaler()),
        ("modelo", LogisticRegression(max_iter=1000)),
    ])
    resultado = cross_validate(modelo, X_bc_full_train[colunas], y_bc_full_train, cv=skf,
                               scoring=["accuracy", "recall"])
    modelo.fit(X_bc_full_train[colunas], y_bc_full_train)
    comparacao[nome] = {
        "Acurácia (validação cruzada)": resultado["test_accuracy"].mean(),
        "Recall maligno (validação cruzada)": resultado["test_recall"].mean(),
        "Acurácia (teste)": accuracy_score(y_bc_test, modelo.predict(X_bc_test[colunas])),
    }

pd.DataFrame(comparacao).T.round(3)

- Com quase metade das *features*, as métricas ficam próximas das originais: a redundância foi removida, a informação ficou.

## **3.5 Síntese**

- **PCA:** sem rótulos, a principal direção de variação separa benignos de malignos.
- **K-means:** dois grupos que coincidem com o diagnóstico em 91% dos nódulos.
- **Clusterização hierárquica:** 30 *features* em 16 blocos; as representantes mantêm o desempenho do classificador.
- Aqui, a principal fonte de variação é a própria doença. Em dados ômicos, ela pode ser outra (lote, tecido, sexo), e os grupos podem não coincidir com a pergunta biológica.